In [ ]:
import random
import numpy as np
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback

from agent import FloodWarningEnv

RUN_ID = "nonorm"

# Seeds
SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training environment
env = make_vec_env(FloodWarningEnv, n_envs=8, seed=SEED)

# Evaluation environment
eval_env = FloodWarningEnv()
eval_env.reset(seed=SEED)

# Agent setup
model = PPO(
    "MlpPolicy",# MLP as the policy, takes observation vector as input and outputs probability distribution over the 4 warning levels
    env,
    learning_rate=3e-4, # how fast network weights update
    n_steps=2048, # how many steps collected across all environments before a policy update (total experience per update = 2048 * 8 = 16384 samples)
    batch_size=64, # how many samples used in each gradient update within a single policy update cycle
    n_epochs=10, # how many times collected experience is reused for gradient updates before being discarded
    gamma=0.99, # discount factor for future rewards
    verbose=1, # prints training progress
    seed=SEED, # seed for reproducibility
    tensorboard_log=f"./logs/{RUN_ID}/"
)

# Evaluation and model saving during training
eval_callback = EvalCallback( # periodically pauses training, runs agent deterministically on separate evaluation environment, saves best performing model
    eval_env,
    best_model_save_path=f"./models/{RUN_ID}/best_model-SEED1/",
    log_path=f"./logs/{RUN_ID}/",
    eval_freq=10_000, # evaluation runs every 10000 steps
    n_eval_episodes=100, # number of episodes to evaluate over for stable performance estimate
    deterministic=True,
)

# Executes training, with evaluation running at specified frequency
model.learn(
    total_timesteps=1_000_000, # runs training loop for 1 million environment steps
    callback=eval_callback
)

# Save final model weights at end of training for resuming training if needed
model.save(f"./models/{RUN_ID}/final_model-SEED1")